In [0]:
%pip install yfinance
%pip install pandas

In [0]:
import time
import random
import yfinance as yf
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import Row

# -------------------------------------
# Config
# -------------------------------------
MAX_WORKERS = 4
BATCH_SIZE = 500
MAX_RETRIES = 5
BASE_DELAY = 2.0

TARGET_TABLE = "finance_intelligence_hub.bronze.stocks_info"

# -------------------------------------
# Read tickers
# -------------------------------------
companies_df = spark.table("finance_intelligence_hub.bronze.companies")
all_tickers = [
    row["ticker"]
    for row in companies_df.select("ticker").distinct().collect()
]
print(f"Total Tickers : {len(all_tickers)}")

# -------------------------------------
# Resume support: skip tickers already loaded
# -------------------------------------
table_exists = spark.catalog.tableExists(TARGET_TABLE)
if table_exists:
    already_done = {
        row["ticker"]
        for row in spark.table(TARGET_TABLE).select("ticker").distinct().collect()
    }
    tickers = [t for t in all_tickers if t not in already_done]
    print(f"Already loaded: {len(already_done)} | Remaining: {len(tickers)}")
else:
    tickers = all_tickers

# -------------------------------------
# Worker with retry + backoff + jitter
# -------------------------------------
def fetch_stock_info(symbol):
    for attempt in range(MAX_RETRIES):
        try:
            ticker = yf.Ticker(symbol)
            info = ticker.info
            return Row(
                ticker=symbol,
                symbol=info.get("symbol"),
                company_name=info.get("longName"),
                short_name=info.get("shortName"),
                exchange=info.get("exchange"),
                currency=info.get("currency"),
                country=info.get("country"),
                sector=info.get("sector"),
                industry=info.get("industry"),
                market_cap=info.get("marketCap"),
                shares_outstanding=info.get("sharesOutstanding"),
                quote_type=info.get("quoteType"),
                dwh_loaded_at=datetime.now(timezone.utc),
            )
        except Exception as e:
            msg = str(e)
            if "Too Many Requests" in msg or "Rate limited" in msg:
                delay = (BASE_DELAY * (2 ** attempt)) + random.uniform(0, 1.5)
                time.sleep(delay)
                continue
            else:
                print(f"{symbol} -> {e}")
                return None
    print(f"{symbol} -> gave up after {MAX_RETRIES} retries (rate limited)")
    return None

# -------------------------------------
# Incremental save helper
# -------------------------------------
table_schema = spark.table(TARGET_TABLE).schema

def flush_batch(rows):
    if not rows:
        return
    batch_df = spark.createDataFrame(rows, schema=table_schema)
    (
        batch_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(TARGET_TABLE)
    )
    print(f"  -> flushed {len(rows)} rows to {TARGET_TABLE}")

# -------------------------------------
# Parallel download with incremental flush every BATCH_SIZE
# -------------------------------------
buffer = []
completed = 0
total = len(tickers)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_stock_info, t): t for t in tickers}

    for future in as_completed(futures):
        row = future.result()
        if row:
            buffer.append(row)
        completed += 1

        if completed % BATCH_SIZE == 0:
            print(f"{completed}/{total} completed")
            flush_batch(buffer)
            buffer = []

flush_batch(buffer)

print("Done.")